In [0]:
#Reading the data from bronze
from pyspark.sql import functions as F
from pyspark.sql.functions import col, regexp_replace, when, row_number
from pyspark.sql.window import Window

df_event = spark.readStream.table("retrailrocket.bronze.events_stream")
df_category = spark.read.table("retrailrocket.bronze.category_tree")
df_items = spark.read.table("retrailrocket.bronze.item_properties_raw")
        

In [0]:
#Clean the events stream
df_event_clean = (df_event
    .select("event", "visitorid", "timestamp", "itemid", "transactionid")
    .withColumn("visitorid", F.col("visitorid").cast("int"))
    .withColumn("itemid", F.col("itemid").cast("int"))
    .withColumn("event_timestamp", F.to_timestamp((F.col("timestamp") / 1000).cast("double")))
)

In [0]:
#Clean item properties: keep usable properties, strip 'n' hash prefix, cast numeric
cleaned_props = (df_items
    .filter(col("property").isin(["categoryid", "available", "790"]))
    .withColumn("value_clean", regexp_replace(col("value"), "^n", "").cast("float"))
    .withColumn("property_name", when(col("property") == "790", "price").otherwise(col("property")))
    .withColumn("itemid", col("itemid").cast("int"))
)

In [0]:
# Keep only the MOST RECENT snapshot per itemid + property (no fan-out history)
window_spec = Window.partitionBy("itemid", "property_name").orderBy(col("timestamp").desc())

current_props = (cleaned_props
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
#Pivot into ONE row per item: price, available, categoryid as real columns
item_dim = (current_props
    .groupBy("itemid")
    .pivot("property_name", ["price", "available", "categoryid"])
    .agg(F.first("value_clean"))
)

In [0]:
#Attach category hierarchy at the item level (not the property-row level)
item_dim = (item_dim
    .withColumn("categoryid", F.col("categoryid").cast("int"))
    .join(
        df_category.select(
            F.col("categoryid").cast("int").alias("cat_categoryid"),
            F.col("parentid")
        ),
        item_dim["categoryid"] == F.col("cat_categoryid"),
        "left"
    )
    .drop("cat_categoryid")
)

In [0]:
df_join = df_event_clean.join(item_dim, on="itemid", how="left")
print(df_join.isStreaming) 

In [0]:
#dbutils.fs.rm("/Volumes/retrailrocket/landing/raw/Streaming/Checkpoint/silver/cleaned_events", recurse=True)

In [0]:
query = (
    df_join.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "/Volumes/retrailrocket/landing/raw/Streaming/Checkpoint/silver/cleaned_events_v2"
        )
        .trigger(processingTime="10 seconds")
        .toTable("retrailrocket.silver.cleaned_events")
)

In [0]:
%sql 
select count(*) from retrailrocket.silver.cleaned_events